# 02 - Training MLP Talky

Notebook ini melatih MLP untuk mengenali huruf SIBI statis (A-Y tanpa J) dari 63 angka landmark.

Alur:
1. Baca daftar CSV eksplisit (`data/dataset.csv` + opsional dataset luar)
2. Bagi data **berdasarkan `person_id`** (train / validasi / test; test tetap rekaman sendiri)
3. Latih MLP, bandingkan beberapa pengaturan (neuron, dropout, learning rate)
4. Evaluasi: akurasi, Confusion Matrix, precision/recall/F1 per huruf
5. Learning curve (25%, 50%, 75%, 100% data)
6. Simpan model dan `labels.txt` (untuk `03_export_tflite.py`)

**Aturan:**
- `normalize()` tidak dipakai atau diubah di sini; data di CSV sudah ternormalisasi.
- Kolom `person_id` hanya dipakai untuk membagi data, lalu dibuang. Input model tetap 63 angka.
- Tambah dataset luar lewat daftar CSV eksplisit (`CSV_FILES`), jangan pola `dataset*.csv`.
- Data **test** (dan validasi bila cukup orang) tetap berasal dari **rekaman sendiri**, bukan dataset luar.
- Semua angka untuk laporan harus berasal dari hasil jalannya notebook ini. Jangan mengarang angka.
- Jalankan dari folder proyek (yang berisi `talky_common.py`), dengan kernel venv proyek.

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras

# supaya talky_common.py (di folder proyek) bisa diimpor
sys.path.insert(0, os.getcwd())
from talky_common import FEATURE_COLUMNS, FOLDER

# Daftar CSV eksplisit (JANGAN pakai pola dataset*.csv).
# File pertama: rekaman sendiri. File bernama 'dataset_luar_*' dianggap dataset luar (khusus train).
# Untuk uji banding (A: data sendiri saja vs B: data sendiri + luar),
# aktifkan atau komentar baris dataset_luar_sibi1.csv lalu Run All.
CSV_FILES = [
    os.path.join(FOLDER, "data", "dataset.csv"),
    os.path.join(FOLDER, "data", "dataset_luar_combinesibi.csv"),
    os.path.join(FOLDER, "data", "dataset_luar_sibi1.csv"),
]
MODEL_DIR = os.path.join(FOLDER, "model")

# ---------------- Pengaturan ----------------
SEED = 42
MAX_EPOCHS = 100
BATCH = 64
PATIENCE = 10          # EarlyStopping: berhenti jika val_loss tidak membaik selama ini

# Pembagian per orang. Biarkan None agar otomatis (jika ada >= 3 orang),
# isi list nama mis. ["nathan"], atau "stratified_10" (4 orang di train + 10% val stratified).
TEST_PERSONS = ["zidna"]
VAL_PERSONS = "stratified_10"
USE_AUGMENTATION = True  # Augmentasi rotasi landmark (+/- 12 deg) + jitter khusus data train
# --------------------------------------------

keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)


TensorFlow: 2.21.0 | Keras: 3.15.1


## 1. Baca data

In [2]:
bagian = []
for path in CSV_FILES:
    assert os.path.exists(path), f"File CSV tidak ditemukan: {path}"
    d = pd.read_csv(path)
    kolom_ok = list(d.columns) == ["person_id", "label", "handedness"] + FEATURE_COLUMNS
    nama = os.path.basename(path)
    print(f"Baca {nama}: {len(d)} baris | Kolom sesuai: {kolom_ok}")
    assert kolom_ok, f"Struktur kolom {nama} tidak sesuai. Jalankan check_data.py dulu."
    d["sumber"] = nama
    d["dari_luar"] = nama.lower().startswith("dataset_luar")
    bagian.append(d)

df = pd.concat(bagian, ignore_index=True)
print(f"\nTotal gabungan: {len(df)} baris dari {len(CSV_FILES)} file")

# J dan Z tidak ikut MLP (ditangani gerak_jz.py)
n_awal = len(df)
df = df[~df["label"].isin(["J", "Z"])].reset_index(drop=True)
if len(df) != n_awal:
    print(f"{n_awal - len(df)} baris J/Z dibuang dari training.")

bentrok = df.groupby("person_id")["dari_luar"].nunique()
assert not (bentrok > 1).any(), (
    f"person_id dipakai di data sendiri sekaligus luar: {list(bentrok[bentrok > 1].index)}"
)

print("\nSampel per sumber:")
print(df.groupby(["sumber", "dari_luar"])["label"].agg(baris="count", huruf="nunique").to_string())
print("\nSampel per orang:")
print(df["person_id"].value_counts().to_string())
print("\nSampel per huruf per orang:")
print(pd.crosstab(df["label"], df["person_id"]).to_string())

Baca dataset.csv: 12000 baris | Kolom sesuai: True
Baca dataset_luar_combinesibi.csv: 1022 baris | Kolom sesuai: True
Baca dataset_luar_sibi1.csv: 2372 baris | Kolom sesuai: True

Total gabungan: 15394 baris dari 3 file

Sampel per sumber:
                                        baris  huruf
sumber                       dari_luar              
dataset.csv                  False      12000     24
dataset_luar_combinesibi.csv True        1022     24
dataset_luar_sibi1.csv       True        2372     24

Sampel per orang:
person_id
sheva          2400
nathan         2400
hafiid         2400
zidna          2400
mayang         2400
sibi1          2372
combinesibi    1022

Sampel per huruf per orang:
person_id  combinesibi  hafiid  mayang  nathan  sheva  sibi1  zidna
label                                                              
A                   50     100     100     100    100    120    100
B                   51     100     100     100    100    117    100
C                   31   

## 2. Bagi data berdasarkan `person_id`

- **3 orang perekam sendiri atau lebih:** test = satu orang, validasi = satu orang, sisanya + dataset luar = train. Akurasi test = performa pada orang yang **belum pernah dilihat** model. Inilah angka yang layak dilaporkan.
- **2 orang perekam sendiri:** test = satu orang; validasi = 15% acak dari orang sendiri lainnya; dataset luar hanya masuk ke train. Validasi jadi optimistis, tetapi akurasi test tetap per orang.
- **1 orang:** pembagian acak. Angka test **optimistis dan tidak boleh dilaporkan** sebagai akurasi model; hanya untuk mengecek bahwa alur kerjanya jalan.

In [3]:
orang_sendiri = sorted(df.loc[~df["dari_luar"], "person_id"].unique())
orang_luar = sorted(df.loc[df["dari_luar"], "person_id"].unique())
print("Orang rekaman sendiri:", orang_sendiri)
if orang_luar:
    print("Kelompok dataset luar (khusus train):", orang_luar)

test_pilihan = [TEST_PERSONS] if isinstance(TEST_PERSONS, str) else (list(TEST_PERSONS) if TEST_PERSONS else None)
val_pilihan = [VAL_PERSONS] if isinstance(VAL_PERSONS, str) else (list(VAL_PERSONS) if VAL_PERSONS else None)

if test_pilihan and any(p in orang_luar for p in test_pilihan):
    print("PERINGATAN: TEST_PERSONS berisi dataset luar! Test seharusnya rekaman sendiri.")

if test_pilihan and val_pilihan:
    MODE = "per_orang"
elif len(orang_sendiri) >= 3:
    if not test_pilihan:
        calon_test = [p for p in orang_sendiri if not val_pilihan or p not in val_pilihan]
        if calon_test:
            test_pilihan = [calon_test[-1]]
    sisa_orang = [p for p in orang_sendiri if not test_pilihan or p not in test_pilihan]
    if not val_pilihan and sisa_orang:
        val_pilihan = [sisa_orang[-1]]
    MODE = "per_orang" if (test_pilihan and val_pilihan) else "dua_orang"
elif len(orang_sendiri) == 2:
    if not test_pilihan:
        test_pilihan = [orang_sendiri[-1]]
    MODE = "dua_orang"
else:
    if not orang_sendiri:
        print("PERINGATAN: tidak ada rekaman sendiri di CSV_FILES.")
    MODE = "satu_orang"

if val_pilihan == ["stratified_10"]:
    MODE = "per_orang"
    df_test = df[df["person_id"].isin(test_pilihan)]
    sisa_sendiri = df[(~df["dari_luar"]) & (~df["person_id"].isin(test_pilihan))]
    df_train_sendiri, df_val = train_test_split(
        sisa_sendiri, test_size=0.10, stratify=sisa_sendiri["label"], random_state=SEED
    )
    df_luar = df[df["dari_luar"] & (~df["person_id"].isin(test_pilihan))]
    df_train = pd.concat([df_train_sendiri, df_luar], ignore_index=True)
    print(f"Mode: 4 orang train (stratified 10% val) | test={test_pilihan} | train={sorted(sisa_sendiri['person_id'].unique())}")
elif MODE == "per_orang":
    pakai = test_pilihan + val_pilihan
    df_test = df[df["person_id"].isin(test_pilihan)]
    df_val = df[df["person_id"].isin(val_pilihan)]
    df_train = df[~df["person_id"].isin(pakai)]
    print(f"Mode: per orang | test={test_pilihan} | validasi={val_pilihan} | train={sorted(df_train['person_id'].unique())}")
elif MODE == "dua_orang":
    df_test = df[df["person_id"].isin(test_pilihan)]
    sisa_sendiri = df[(~df["dari_luar"]) & (~df["person_id"].isin(test_pilihan))]
    df_train_sendiri, df_val = train_test_split(sisa_sendiri, test_size=0.15, stratify=sisa_sendiri["label"], random_state=SEED)
    df_luar = df[df["dari_luar"] & (~df["person_id"].isin(test_pilihan))]
    df_train = pd.concat([df_train_sendiri, df_luar], ignore_index=True)
    print(f"Mode: dua orang | test={test_pilihan} | validasi = 15% acak dari rekaman sendiri lainnya")
    if len(df_luar) > 0:
        print(f"  Train gabungan: rekaman sendiri={len(df_train_sendiri)} + data luar={len(df_luar)}")
    print("PERINGATAN: akurasi validasi optimistis. Tambah orang perekam sendiri (minimal 3).")
else:
    df_sumber = df[~df["dari_luar"]] if orang_sendiri else df
    df_trval, df_test = train_test_split(df_sumber, test_size=0.15, stratify=df_sumber["label"], random_state=SEED)
    df_train_s, df_val = train_test_split(df_trval, test_size=0.1765, stratify=df_trval["label"], random_state=SEED)
    df_luar = df[df["dari_luar"]] if orang_sendiri else df.iloc[0:0]
    df_train = pd.concat([df_train_s, df_luar], ignore_index=True)
    print("Mode: satu orang (acak)")
    print("PERINGATAN: angka test OPTIMISTIS dan TIDAK boleh dilaporkan sebagai akurasi model.")

assert len(df_train) > 0 and len(df_val) > 0 and len(df_test) > 0, "Salah satu himpunan train/val/test kosong!"
TEST_AKTIF = test_pilihan if MODE != "satu_orang" else ["15% acak"]
print(f"\nTrain: {len(df_train)} | Validasi: {len(df_val)} | Test: {len(df_test)}")

Orang rekaman sendiri: ['hafiid', 'mayang', 'nathan', 'sheva', 'zidna']
Kelompok dataset luar (khusus train): ['combinesibi', 'sibi1']
Mode: 4 orang train (stratified 10% val) | test=['zidna'] | train=['hafiid', 'mayang', 'nathan', 'sheva']

Train: 12034 | Validasi: 960 | Test: 2400


In [4]:
# Label -> angka. person_id, handedness, dan sumber dibuang; input model hanya 63 angka.
le = LabelEncoder()
le.fit(sorted(df["label"].unique()))
kelas = list(le.classes_)
n_kelas = len(kelas)
print("Kelas:", n_kelas, kelas)

def siapkan(d):
    X = d[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    y = le.transform(d["label"])
    return X, y

X_train, y_train = siapkan(df_train)
X_val, y_val = siapkan(df_val)
X_test, y_test = siapkan(df_test)

# Augmentasi khusus data train: rotasi 2D (+/- 12 deg), yaw halus (+/- 6 deg), & jitter.
# X_val dan X_test TIDAK disentuh agar evaluasi tetap murni dan jujur.
def augment_landmarks(X, y, n_copies=2, max_roll_deg=12.0, max_yaw_deg=6.0, noise_std=0.005, seed=42):
    rng = np.random.default_rng(seed)
    X_list = [X]
    y_list = [y]
    for _ in range(n_copies):
        pts = X.reshape(-1, 21, 3).copy()
        # 1) Rotasi 2D roll (sumbu Z) mengelilingi pergelangan (0, 0)
        roll = np.deg2rad(rng.uniform(-max_roll_deg, max_roll_deg, size=(len(pts), 1)))
        cr, sr = np.cos(roll), np.sin(roll)
        x0, y0 = pts[:, :, 0].copy(), pts[:, :, 1].copy()
        pts[:, :, 0] = x0 * cr - y0 * sr
        pts[:, :, 1] = x0 * sr + y0 * cr
        # 2) Rotasi yaw halus (sumbu Y) untuk variasi sudut serong tangan
        yaw = np.deg2rad(rng.uniform(-max_yaw_deg, max_yaw_deg, size=(len(pts), 1)))
        cy, sy = np.cos(yaw), np.sin(yaw)
        x1, z1 = pts[:, :, 0].copy(), pts[:, :, 2].copy()
        pts[:, :, 0] = x1 * cy + z1 * sy
        pts[:, :, 2] = -x1 * sy + z1 * cy
        # 3) Jitter halus pada titik jari (pergelangan titik 0 tetap 0,0,0)
        noise = rng.normal(0.0, noise_std, size=pts.shape).astype(np.float32)
        noise[:, 0, :] = 0.0
        pts = pts + noise
        # 4) Jaga skala maksimum = 1.0 sesuai normalisasi kanonikal
        norms = np.max(np.linalg.norm(pts, axis=2), axis=1, keepdims=True)[:, :, None]
        pts = pts / np.maximum(norms, 1e-6)
        X_list.append(pts.reshape(-1, 63).astype(np.float32))
        y_list.append(y.copy())
    return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)

if USE_AUGMENTATION:
    X_train, y_train = augment_landmarks(X_train, y_train, n_copies=2, seed=SEED)
    print(f"Augmentasi aktif: X_train menjadi {X_train.shape} (asli + 2x rotasi & jitter)")

print("Bentuk X_train:", X_train.shape, "(harus 63 kolom)")

# Menyeimbangkan huruf: bobot kelas (data asli tidak dihapus)
def hitung_cw(y):
    ada = np.unique(y)
    bobot = compute_class_weight("balanced", classes=ada, y=y)
    return {int(k): float(v) for k, v in zip(ada, bobot)}

hilang = [kelas[i] for i in range(n_kelas) if i not in set(y_train)]
if hilang:
    print("PERINGATAN: huruf ini tidak ada di data train:", hilang)

Kelas: 24 [np.str_('A'), np.str_('B'), np.str_('C'), np.str_('D'), np.str_('E'), np.str_('F'), np.str_('G'), np.str_('H'), np.str_('I'), np.str_('K'), np.str_('L'), np.str_('M'), np.str_('N'), np.str_('O'), np.str_('P'), np.str_('Q'), np.str_('R'), np.str_('S'), np.str_('T'), np.str_('U'), np.str_('V'), np.str_('W'), np.str_('X'), np.str_('Y')]
Augmentasi aktif: X_train menjadi (36102, 63) (asli + 2x rotasi & jitter)
Bentuk X_train: (36102, 63) (harus 63 kolom)


## 3. Model dan pelatihan

Arsitektur awal: `Input(63) -> Dense(128, ReLU) -> Dropout -> Dense(64, ReLU) -> Dropout -> Dense(n_kelas, Softmax)`.

In [5]:
def build_model(n1=128, n2=64, dropout=0.3, lr=0.001):
    model = keras.Sequential([
        keras.layers.Input(shape=(63,)),
        keras.layers.Dense(n1, activation="relu"),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(n2, activation="relu"),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(n_kelas, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model


def latih(model, X_tr, y_tr, cw=None):
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                       restore_best_weights=True)
    return model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                     epochs=MAX_EPOCHS, batch_size=BATCH,
                     class_weight=cw, callbacks=[es], verbose=0)


model_contoh = build_model()
model_contoh.summary()

Model: "sequential"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴─────────────────────

## 4. Eksperimen pengaturan

Setiap eksperimen dicatat dalam tabel (untuk bab pengujian). Pengaturan terbaik dipilih berdasarkan **akurasi validasi**, bukan akurasi test, supaya test tetap jujur.

In [6]:
konfigurasi = [
    dict(n1=128, n2=64,  dropout=0.3, lr=0.001),    # awal
    dict(n1=64,  n2=32,  dropout=0.3, lr=0.001),
    dict(n1=256, n2=128, dropout=0.3, lr=0.001),
    dict(n1=128, n2=64,  dropout=0.2, lr=0.001),
    dict(n1=128, n2=64,  dropout=0.5, lr=0.001),
    dict(n1=128, n2=64,  dropout=0.3, lr=0.0005),
    dict(n1=128, n2=64,  dropout=0.3, lr=0.003),
]

catatan = []
terbaik = {"val_acc": -1, "val_loss": float("inf")}
for i, cfg in enumerate(konfigurasi, 1):
    keras.utils.set_random_seed(SEED)
    m = build_model(**cfg)
    h = latih(m, X_train, y_train, hitung_cw(y_train))
    val_loss, val_acc = m.evaluate(X_val, y_val, verbose=0)
    test_acc = m.evaluate(X_test, y_test, verbose=0)[1]
    catatan.append({"no": i, **cfg, "epoch": len(h.history["loss"]),
                    "val_acc": round(val_acc, 4), "test_acc": round(test_acc, 4)})
    print(f"[{i}/{len(konfigurasi)}] {cfg} -> val {val_acc:.4f} (loss {val_loss:.4f}) | test {test_acc:.4f} | epoch {len(h.history['loss'])}")
    if (val_acc > terbaik["val_acc"]) or (val_acc == terbaik["val_acc"] and val_loss < terbaik["val_loss"]):
        terbaik = {"val_acc": val_acc, "val_loss": val_loss, "cfg": cfg, "model": m, "history": h, "no": i}

tabel = pd.DataFrame(catatan)
os.makedirs(MODEL_DIR, exist_ok=True)
tabel.to_csv(os.path.join(MODEL_DIR, "eksperimen.csv"), index=False)
print(f"\nPengaturan terbaik (berdasarkan validasi): eksperimen no. {terbaik['no']} -> {terbaik['cfg']}")
tabel

[1/7] {'n1': 128, 'n2': 64, 'dropout': 0.3, 'lr': 0.001} -> val 0.9937 (loss 0.0229) | test 0.7796 | epoch 55
[2/7] {'n1': 64, 'n2': 32, 'dropout': 0.3, 'lr': 0.001} -> val 0.9792 (loss 0.0672) | test 0.7650 | epoch 48
[3/7] {'n1': 256, 'n2': 128, 'dropout': 0.3, 'lr': 0.001} -> val 0.9927 (loss 0.0151) | test 0.7875 | epoch 50
[4/7] {'n1': 128, 'n2': 64, 'dropout': 0.2, 'lr': 0.001} -> val 0.9958 (loss 0.0139) | test 0.7796 | epoch 76
[5/7] {'n1': 128, 'n2': 64, 'dropout': 0.5, 'lr': 0.001} -> val 0.9844 (loss 0.0436) | test 0.7721 | epoch 77
[6/7] {'n1': 128, 'n2': 64, 'dropout': 0.3, 'lr': 0.0005} -> val 0.9927 (loss 0.0295) | test 0.7812 | epoch 57
[7/7] {'n1': 128, 'n2': 64, 'dropout': 0.3, 'lr': 0.003} -> val 0.9865 (loss 0.0375) | test 0.7704 | epoch 30

Pengaturan terbaik (berdasarkan validasi): eksperimen no. 4 -> {'n1': 128, 'n2': 64, 'dropout': 0.2, 'lr': 0.001}


In [7]:
# Kurva belajar model terbaik
h = terbaik["history"].history
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h["loss"], label="train"); ax[0].plot(h["val_loss"], label="validasi")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(h["accuracy"], label="train"); ax[1].plot(h["val_accuracy"], label="validasi")
ax[1].set_title("Akurasi"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

<string>:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 5. Evaluasi model terbaik pada data test

In [8]:
model_final = terbaik["model"]
proba = model_final.predict(X_test, verbose=0)
pred = proba.argmax(axis=1)
akurasi_test = float((pred == y_test).mean())
print(f"Akurasi test: {akurasi_test:.4f}  (mode pembagian: {MODE}, test = {TEST_AKTIF})")
if MODE != "per_orang" or any(p in orang_luar for p in TEST_AKTIF):
    print("PERINGATAN: mode bukan per_orang dengan >= 3 orang sendiri; angka ini belum layak dilaporkan.")

print("\nPrecision / recall / F1 per huruf:")
print(classification_report(y_test, pred, labels=list(range(n_kelas)),
                            target_names=kelas, zero_division=0))

Akurasi test: 0.7796  (mode pembagian: per_orang, test = ['zidna'])

Precision / recall / F1 per huruf:
              precision    recall  f1-score   support

           A       0.71      1.00      0.83       100
           B       0.81      0.79      0.80       100
           C       0.85      0.64      0.73       100
           D       0.94      0.74      0.83       100
           E       0.93      0.53      0.68       100
           F       1.00      0.66      0.80       100
           G       0.93      0.64      0.76       100
           H       0.57      0.91      0.70       100
           I       0.97      0.89      0.93       100
           K       0.73      0.91      0.81       100
           L       0.91      0.53      0.67       100
           M       0.64      0.77      0.70       100
           N       0.82      0.80      0.81       100
           O       0.76      0.79      0.77       100
           P       0.63      1.00      0.78       100
           Q       0.82      0.

In [9]:
cm = confusion_matrix(y_test, pred, labels=list(range(n_kelas)))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=kelas, yticklabels=kelas)
plt.xlabel("Tebakan"); plt.ylabel("Huruf asli"); plt.title("Confusion Matrix (data test)")
plt.tight_layout(); plt.show()

# Pasangan huruf yang paling sering tertukar
salah = [(kelas[i], kelas[j], int(cm[i, j])) for i in range(n_kelas) for j in range(n_kelas)
         if i != j and cm[i, j] > 0]
salah.sort(key=lambda t: -t[2])
print("Pasangan tertukar terbanyak (asli -> tebakan, jumlah):")
for a, b, n in salah[:10]:
    print(f"  {a} -> {b}: {n}")
if not salah:
    print("  (tidak ada kesalahan pada data test)")

Pasangan tertukar terbanyak (asli -> tebakan, jumlah):
  U -> K: 29
  T -> X: 25
  L -> X: 24
  L -> H: 20
  C -> H: 17
  E -> O: 17
  W -> P: 17
  E -> M: 15
  E -> S: 15
  G -> Q: 15


<string>:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 6. Learning curve

Latih ulang dengan 25%, 50%, 75%, dan 100% data train (pengaturan terbaik), lalu ukur akurasi test. Setiap titik diulang beberapa kali dengan seed berbeda, dilaporkan rata-rata dan simpangan baku. Jika kurva sudah mendatar, menambah data tidak banyak membantu.

In [10]:
fraksi = [0.25, 0.5, 0.75, 1.0]
N_ULANG = 3
cfg_terbaik = terbaik["cfg"]

baris = []
for f in fraksi:
    akurasi = []
    for r in range(N_ULANG):
        if f < 1.0:
            idx, _ = train_test_split(np.arange(len(y_train)), train_size=f,
                                      stratify=y_train, random_state=SEED + r)
        else:
            idx = np.arange(len(y_train))
        keras.utils.set_random_seed(SEED + r)
        m = build_model(**cfg_terbaik)
        latih(m, X_train[idx], y_train[idx], hitung_cw(y_train[idx]))
        akurasi.append(m.evaluate(X_test, y_test, verbose=0)[1])
    baris.append({"data_train": f"{int(f * 100)}%", "n_sampel": int(len(idx)),
                  "test_acc_rata2": round(float(np.mean(akurasi)), 4),
                  "simpangan_baku": round(float(np.std(akurasi)), 4)})
    print(baris[-1])

lc = pd.DataFrame(baris)
lc.to_csv(os.path.join(MODEL_DIR, "learning_curve.csv"), index=False)

plt.figure(figsize=(6, 4))
plt.errorbar([int(x[:-1]) for x in lc["data_train"]], lc["test_acc_rata2"],
             yerr=lc["simpangan_baku"], marker="o", capsize=4)
plt.xlabel("Persentase data train (%)"); plt.ylabel("Akurasi test")
plt.title("Learning curve"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
lc

{'data_train': '25%', 'n_sampel': 9025, 'test_acc_rata2': 0.7463, 'simpangan_baku': 0.0056}
{'data_train': '50%', 'n_sampel': 18051, 'test_acc_rata2': 0.7622, 'simpangan_baku': 0.0074}
{'data_train': '75%', 'n_sampel': 27076, 'test_acc_rata2': 0.7764, 'simpangan_baku': 0.0031}
{'data_train': '100%', 'n_sampel': 36102, 'test_acc_rata2': 0.7788, 'simpangan_baku': 0.0119}


<string>:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 7. Simpan model

Disimpan ke folder `model/`: `talky_mlp.keras`, `labels.txt` (urutan kelas = urutan output model), serta tabel eksperimen dan learning curve. Langkah berikutnya: `03_export_tflite.py`.

In [11]:
model_final.save(os.path.join(MODEL_DIR, "talky_mlp.keras"))
with open(os.path.join(MODEL_DIR, "labels.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(kelas))
print("Tersimpan di:", MODEL_DIR)
print("labels.txt:", kelas)

Tersimpan di: D:\PROJEK GIT\talky-ml\model
labels.txt: [np.str_('A'), np.str_('B'), np.str_('C'), np.str_('D'), np.str_('E'), np.str_('F'), np.str_('G'), np.str_('H'), np.str_('I'), np.str_('K'), np.str_('L'), np.str_('M'), np.str_('N'), np.str_('O'), np.str_('P'), np.str_('Q'), np.str_('R'), np.str_('S'), np.str_('T'), np.str_('U'), np.str_('V'), np.str_('W'), np.str_('X'), np.str_('Y')]


In [12]:
from talky_common import CONNECTIONS

def gambar(ax, huruf, orang, warna):
    d = df[(df["label"] == huruf) & (df["person_id"] == orang)]
    if len(d) == 0:
        ax.set_title(f"{huruf} - {orang} (kosong)")
        ax.axis("off")
        return
    p = d[FEATURE_COLUMNS].mean().to_numpy().reshape(21, 3)
    for a, b in CONNECTIONS:
        ax.plot([p[a, 0], p[b, 0]], [p[a, 1], p[b, 1]], color=warna)
    ax.scatter(p[:, 0], p[:, 1], color=warna, s=15)
    ax.set_title(f"{huruf} - {orang} (n={len(d)})")
    ax.invert_yaxis()
    ax.set_aspect("equal")

orang_plot = sorted(df["person_id"].unique())
huruf_plot = ["E", "W"]
warna_plot = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

fig, axes = plt.subplots(
    len(orang_plot), len(huruf_plot),
    figsize=(4 * len(huruf_plot), 4 * len(orang_plot)),
    squeeze=False,
)
for i, orang in enumerate(orang_plot):
    warna = warna_plot[i % len(warna_plot)]
    for j, huruf in enumerate(huruf_plot):
        gambar(axes[i, j], huruf, orang, warna)
plt.tight_layout()
plt.show()

<string>:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
